# WS 9.1: Regression

In this worksheet, we explore how to **predict** one variable using another. We will work with a dataset of motorcycle prices and build up to a measure of prediction quality called **R-squared** (R²), step by step.

This follows the same arc as WS 6.2, where we measured how spread out data is using variance. Here, we will use variance to measure how much *better* our predictions get when we use additional information.

## Getting the data

Download the motorcycle dataset from [this link](https://drive.google.com/file/d/1QAiBGTJzATWaKBzGp_vQisJPe87Wc_Ak/view?usp=drive_link) and upload it to your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Run the cell below to import the libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

**Exercise 1.1:** Load the data using `pd.read_csv()` with the path to your file on Google Drive. Store it in a variable called `bikes` and display the first 10 rows with `bikes.head(10)`. Which columns are numeric?

In [ ]:
bikes = pd.read_csv("___")  # Fill in your path
bikes.head(10)

*Your answer:*

This dataset contains information on 241 motorcycles from around the world. Each row is one motorcycle, and the columns include:

| Column | Description |
|---|---|
| `Company` | Manufacturer (e.g. Ducati, KTM, Harley-Davidson) |
| `Model` | Model name |
| `Horsepower` | Engine power |
| `CC` | Engine displacement (cubic centimeters) |
| `Torque_Nm` | Torque in Newton-meters |
| `Cylinders` | Number of engine cylinders |
| `Year` | Model year |
| `Seats` | Number of seats (1 or 2) |
| `Price_millions_VND` | Price in millions of Vietnamese dong |

Prices range from about 9 million VND (a small scooter) to over 1,200 million VND (a premium touring bike).

# Part 1: Predicting with No Information

Imagine someone asks you to predict the price of a motorcycle, but you know nothing about it — not the brand, not the engine size, nothing. You just have to pick one number.

In class, you tried this and saw that the **mean** is the best single-number prediction. But how good is "best"? We need a way to measure how far off our predictions are. In WS 6.2, we used **variance** to measure how spread out data is around the mean. Here, we will use the same idea — but now we are thinking of the spread as *prediction error*.

**Exercise 1.2:** Compute the mean and variance of `bikes["Price_millions_VND"]`.

In [ ]:
# Your code here

Recall from WS 7.2 that `plt.hist()` creates a histogram:

```python
plt.hist(data, bins=25)
plt.xlabel("label")
plt.title("title")
plt.show()
```

**Exercise 1.3:** Create a histogram of `bikes["Price_millions_VND"]` with `bins=25`. Label the x-axis `"Price (millions VND)"`.

In [ ]:
# Your code here

If you used the mean as your prediction for every bike, the **residual** for each bike would be:

$$\text{residual} = \text{actual price} - \text{predicted price}$$

When the prediction is the mean, this is exactly the **deviation from the mean** that you computed in WS 6.2 Part 2.

**Exercise 1.4:** Compute the residuals when predicting the mean for every bike. Store them in a variable called `residuals_mean`.

*Hint:* Subtract `bikes["Price_millions_VND"].mean()` from `bikes["Price_millions_VND"]`.

In [ ]:
# Your code here

In WS 7.2, you used `plt.subplot()` to place multiple plots side by side:

```python
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(data1, bins=25, range=(xmin, xmax))
plt.title("Title 1")

plt.subplot(1, 2, 2)
plt.hist(data2, bins=25, range=(xmin, xmax))
plt.title("Title 2")

plt.tight_layout()
plt.show()
```

Using the same `range=` in both histograms puts them on the same scale so you can compare the spread visually.

**Exercise 1.5:** Create two histograms side by side: the original prices on the left and the residuals on the right. Use the same range for both so you can compare the spread.

*Hint:* For the range, you can use `range=(-500, 1300)` to cover both distributions.

In [ ]:
# Your code here

**Exercise 1.6:** Compare the two histograms. How does the shape of the residuals compare to the shape of the prices? What is different about where they are centered?

*Your answer:*

**Exercise 1.7:** Compute `residuals_mean.var()`. How does it compare to `bikes["Price_millions_VND"].var()`?

In [ ]:
# Your code here

*Your answer:*

So predicting the mean gives us residuals with the same variance as the original prices. The spread of our errors is just as wide as the spread of the data itself. Can we do better?

# Part 2: Using Horsepower to Predict

Instead of predicting the same price for every bike, what if we used information about the bike — like its horsepower — to make a better prediction?

Recall from WS 8.1 that `plt.scatter()` creates a scatter plot and `.corr()` computes the correlation coefficient:

```python
plt.scatter(data["column1"], data["column2"])
plt.xlabel("label 1")
plt.ylabel("label 2")
plt.show()
```

**Exercise 2.1:** Create a scatter plot with `Horsepower` on the x-axis and `Price_millions_VND` on the y-axis.

In [ ]:
# Your code here

**Exercise 2.2:** Describe the relationship you see. Is it positive or negative? Strong or weak? Does knowing a bike's horsepower tell you something about its price?

*Your answer:*

The scatter plot shows a trend, and the correlation confirms it. But correlation just tells us *how strong* the relationship is — it does not give us a **prediction**. We want a formula that takes a horsepower value and gives back a predicted price.

NumPy can find the best straight line through the scatter plot:

```python
m, b = np.polyfit(x_data, y_data, 1)
```

This returns the slope (`m`) and intercept (`b`) as two separate values. With these, we can predict the price for any bike:

$$\text{predicted price} = m \times \text{horsepower} + b$$

**Exercise 2.3:** Use `np.polyfit` to find the slope and intercept for predicting price from horsepower. Print both values.

In [ ]:
# Your code here

**Exercise 2.4:** Compute the predicted price for every bike using the formula `m * bikes["Horsepower"] + b`. Store the result in a variable called `predictions`.

In [ ]:
# Your code here

Run the cell below to see the best-fit line on the scatter plot.

In [ ]:
plt.scatter(bikes["Horsepower"], bikes["Price_millions_VND"], alpha=0.5)
hp_range = np.array([bikes["Horsepower"].min(), bikes["Horsepower"].max()])
plt.plot(hp_range, m * hp_range + b, color="red")
plt.xlabel("Horsepower")
plt.ylabel("Price (millions VND)")
plt.title("Price vs Horsepower with Best Fit Line")
plt.show()

Now we can measure how far off each prediction is. The **residual** is the difference between the actual price and the predicted price — just like in Part 1, but now each bike gets its own prediction instead of the mean.

**Exercise 2.5:** Compute the residuals: `bikes["Price_millions_VND"] - predictions`. Store the result in a variable called `residuals`.

In [ ]:
# Your code here

**Exercise 2.6:** Compute the variance of `residuals`. How does it compare to the variance of the prices from Exercise 1.2?

In [ ]:
# Your code here

*Your answer:*

Let's see this visually. In Part 1, you plotted the residuals from predicting the mean. Now let's compare those with the residuals from using horsepower.

**Exercise 2.7:** Create two histograms side by side: `residuals_mean` on the left and `residuals` (from the horsepower model) on the right. Use the same range for both (e.g., `range=(-500, 1000)`).

In [ ]:
# Your code here

**Exercise 2.8:** Compare the two histograms. What happened to the spread of the residuals when we used horsepower to predict?

*Your answer:*

# Part 3: Measuring Prediction Quality

We have seen — both in the numbers and the histograms — that using horsepower reduces the variance of our residuals. But by how much? We need a single number that summarizes how good our prediction is.

**Exercise 3.1:** Divide `residuals.var()` by `bikes["Price_millions_VND"].var()`. This ratio is the **error fraction** — the proportion of the original spread that remains in our prediction errors.

What does your result mean in plain language?

In [ ]:
# Your code here

*Your answer:*

The error fraction tells us what proportion of the spread is *left over*. But in practice, it is more natural to talk about what our model *did* capture, so that higher numbers mean better predictions — like a test score. We flip the error fraction by subtracting it from 1:

$$R^2 = 1 - \frac{\text{Var(residuals)}}{\text{Var(prices)}}$$

This is called **R-squared** (R²). If the error fraction is 0.15, then R² = 1 − 0.15 = 0.85 — our model captures 85% of the variance in prices.

**Exercise 3.2:** Compute R² for the horsepower model.

In [ ]:
# Your code here

**Exercise 3.3:** What would R² = 0 mean? What about R² = 1?

*Your answer:*

### Why variance?

In WS 6.2, we noted that variance is in squared units, which makes it harder to interpret directly than standard deviation. But variance has a property that makes R² work cleanly.

**Exercise 3.4:** Compute and display the variance of the prices, the variance of the predictions, and the variance of the residuals. (You have already calculated some of these quantities above.) Then add up the variance of the predictions and the variance of the residuals. What do you notice?

In [ ]:
# Your code here

*Your answer:*

**Exercise 3.5:** Now try the same thing using `.std()` instead of `.var()`. Does the standard deviation of predictions plus the standard deviation of residuals equal the standard deviation of the prices?

In [ ]:
# Your code here

*Your answer:*

**Note:** This clean split — Var(prices) = Var(predictions) + Var(residuals) — works because `np.polyfit` finds the line that minimizes squared errors. This is what makes the variances add up so neatly. Other prediction methods may not have this property, but R² (defined as 1 − error fraction) is still a useful measure of prediction quality regardless.

### A connection to correlation

Correlation measures the strength of a linear relationship. Regression uses that relationship to make predictions. It turns out these are two views of the same thing.

**Exercise 3.6:** Compute the correlation (also known as "R") between `Horsepower` and `Price_millions_VND`, then square it. Compare the result to R².

In [ ]:
# Your code here

*Your answer:*

# Part 4: Adding a Second Predictor

What if we used *two* variables to predict price — does adding more information improve our prediction?

With one predictor, our formula was:

$$\text{price} = m \times \text{horsepower} + b$$

With two predictors, it becomes:

$$\text{price} = m_1 \times x_1 + m_2 \times x_2 + b$$

So instead of one slope and one intercept, we now have **two slopes** and one intercept — three values total.

### Returning multiple values

In Exercise 2.3, you wrote `m, b = np.polyfit(...)`. When a function returns multiple values, Python lets you **unpack** them into separate variables. `np.polyfit` returned two values (slope, intercept), so we unpacked into two variables.

Our new function will return **three** values (two slopes and an intercept), so we unpack into three variables:

```python
m1, m2, b = fit_two_predictors(...)
```

Run the cell below to load the function.

In [ ]:
#@title Run this cell to load fit_two_predictors
def fit_two_predictors(data, target, predictor1, predictor2):
    "Like np.polyfit, but with two predictors. Returns slope1, slope2, intercept."
    subset = data[[target, predictor1, predictor2]].dropna()
    y = subset[target].values
    x1 = subset[predictor1].values
    x2 = subset[predictor2].values
    A = np.column_stack([x1, x2, np.ones(len(y))])
    coefficients, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
    return coefficients[0], coefficients[1], coefficients[2]

You call it like this:

```python
m1, m2, b = fit_two_predictors(bikes, "Price_millions_VND", "Horsepower", "Another Variable")
```

(where "Another Variable" is the name of any other numeric variable in the dataset)

Then the prediction for each bike is:

```python
predictions_2var = m1 * bikes["Horsepower"] + m2 * bikes["Another Variable"] + b
```

This is the same idea as Part 2 — a linear formula — but now with two terms instead of one.

**Exercise 4.1:** Use `fit_two_predictors` to predict price from `Horsepower` and `Torque_Nm`. Unpack the three return values into `m1`, `m2`, `b` and print them. What does each number mean?

In [ ]:
# Your code here

*Your answer:*

**Exercise 4.2:** Compute the predictions and residuals. Then, compute and display the R². How does it compare to the single-variable R² from Part 3?

In [ ]:
# Your code here

*Your answer:*

Here are all numeric columns you can try as a second predictor:

| Column | Description |
|---|---|
| `CC` | Engine displacement (cubic centimeters) |
| `Torque_Nm` | Torque (Newton-meters) |
| `Cylinders` | Number of cylinders |
| `Year` | Model year |
| `Seats` | Number of seats |

**Exercise 4.3:** Choose a *different* second predictor alongside Horsepower. Remember to unpack the three return values from `fit_two_predictors`, then compute predictions and R². Then try `"Year"` or `"Seats"` as the second predictor. Which variables improve the prediction the most? Which ones barely help?

In [ ]:
# Your code here

*Your answer:*

**Exercise 4.4:** Why do you think some variables improve R² and others do not? In your own words, what does R² tell you about a prediction?

*Your answer:*

# Credits

This worksheet was created by Ethan Brown with assistance from [Claude Code](https://claude.ai/code).